In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/processed/games_export.csv')
print(f'Строк: {len(df)}, колонок: {df.shape[1]}')
df.head()

Строк: 125855, колонок: 10


,app_id,name,release_date,estimated_owners,peak_ccu,price,positive,negative,avg_playtime_forever,required_age
0,2539430,Black Dragon Mage Playtest,2023-08-01,0 - 0,0,0.00,0,0,0,0
1,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,0 - 20000,0,5.24,252,3,8,0
2,1034400,Mystery Solitaire The Black Raven,2019-05-06,0 - 20000,0,4.99,21,3,0,0
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,2024-10-31,0 - 20000,1,8.99,0,0,0,0
4,3631080,Maze Quest VR,2025-04-24,0 - 20000,0,4.99,0,0,0,0


## Распределение цен

Анализируем только платные игры (price > 0).
Из 125 855 игр платных — 99 194 (79%).

Ключевые наблюдения:
- Медиана цены: $3.59 — типичная игра стоит около 4 долларов
- Среднее: $6.10 — вдвое выше медианы из-за дорогих выбросов
- 95% каталога стоит до $16.49
- Только 1% игр дороже $34.99

In [2]:
paid = df[df['price'] > 0].copy()

print(f'Платных игр: {len(paid)} ({len(paid)*100/len(df):.1f}% каталога)')
print(f'Медиана цены: ${paid["price"].median():.2f}')
print(f'Средняя цена: ${paid["price"].mean():.2f}')
print()
print('Перцентили цен:')
print(paid['price'].quantile([0.75, 0.90, 0.95, 0.99]).to_string())

Платных игр: 99194 (78.8% каталога)
Медиана цены: $3.59
Средняя цена: $6.10

Перцентили цен:
0.75     6.99
0.90    11.99
0.95    16.49
0.99    34.99


In [3]:
conditions = [
    paid['price']<=5,
    (5<paid['price']) & (paid['price']<20),
    paid['price']>=20
]
choices = ['budget', 'mid', 'premium']
paid['price_segment'] = np.select(conditions, choices, default = 'unknown')
print(paid['price_segment'].value_counts())

price_segment
budget     66830
mid        29780
premium     2584
Name: count, dtype: int64


In [4]:
paid['price_quartile'] = pd.qcut(paid['price'], q=4, labels=['q1', 'q2', 'q3', 'q4'])
paid['segment_avg_price'] = paid.groupby('price_quartile', observed = True)['price'].transform('mean')
paid['price_vs_avg'] = paid['price'] - paid['segment_avg_price'] 
paid[['name', 'price', 'price_quartile', 'segment_avg_price', 'price_vs_avg']].head(20)
_, bins = pd.qcut(paid['price'], q=4, retbins=True)
print('Границы квартилей:')
for i in range(len(bins)-1):
    print(f'Q{i+1}: {bins[i]:.2f} — {bins[i+1]:.2f}')

print('\nКоличество игр в каждом квартиле:')
print(paid['price_quartile'].value_counts().sort_index())

Границы квартилей:
Q1: 0.49 — 1.49
Q2: 1.49 — 3.59
Q3: 3.59 — 6.99
Q4: 6.99 — 999.98

Количество игр в каждом квартиле:
price_quartile
q1    25689
q2    24027
q3    25340
q4    24138
Name: count, dtype: int64


In [5]:
genres = pd.read_csv('data/processed/genres_export.csv')
print(genres.shape)
genres.head()
df_merged = paid.merge(genres, left_on='app_id', right_on='game_id', how='left')
print(f'Строк до merge: {len(paid)}')
print(f'Строк после merge: {len(df_merged)}')

(338583, 2)
Строк до merge: 99194
Строк после merge: 276859


In [6]:
print(df_merged['game_id'].isna().sum())
no_genre = df[df['price'] == 0]
print(f'Бесплатных игр всего: {len(no_genre)}')

109
Бесплатных игр всего: 26661


In [7]:
df_merged.groupby('genre').agg(
    games_count=('app_id', 'nunique'),
    avg_price=('price', 'mean'),
    median_price=('price', 'median')
).round(2).sort_values('avg_price', ascending=False)

,games_count,avg_price,median_price
genre,,,
Video Production,216,24.71,12.99
Animation & Modeling,329,21.20,8.99
Audio Production,166,19.54,9.99
Education,296,16.97,7.99
Software Training,152,16.86,8.99
Design & Illustration,440,16.65,7.99
Web Publishing,77,16.40,9.99
Game Development,226,16.22,9.99
Photo Editing,141,16.21,6.49


In [8]:
genre_counts = df_merged.groupby('genre')['app_id'].nunique().reset_index()
genre_counts.columns = ['genre', 'games_count']
genre_counts = genre_counts.sort_values('games_count', ascending=False)
genre_counts['share'] = genre_counts['games_count']/genre_counts['games_count'].sum()*100
genre_counts['cumulative_share'] = genre_counts['share'].cumsum()
print(genre_counts.round(1).to_string())

                    genre  games_count  share  cumulative_share
15                  Indie        70937   25.6              25.6
6                  Casual        44070   15.9              41.6
2                  Action        39984   14.4              56.0
3               Adventure        39922   14.4              70.4
24             Simulation        21628    7.8              78.2
27               Strategy        19575    7.1              85.3
20                    RPG        18201    6.6              91.9
9            Early Access         9681    3.5              95.4
26                 Sports         4284    1.5              96.9
21                 Racing         3605    1.3              98.2
16  Massively Multiplayer         1152    0.4              98.7
29              Utilities          773    0.3              98.9
7   Design & Illustration          440    0.2              99.1
31                Violent          352    0.1              99.2
4    Animation & Modeling          329  

In [17]:
pivot = df_merged.groupby(['price_segment', 'genre'])['app_id'].nunique().unstack(fill_value=0)
print(pivot)

genre          360 Video  Accounting  Action  Adventure  Animation & Modeling  \
price_segment                                                                   
budget                 1           7   26909      26040                   125   
mid                    0           8   12077      12874                   116   
premium                0           4     998       1008                    88   

genre          Audio Production  Casual  Design & Illustration  Documentary  \
price_segment                                                                 
budget                       58   33090                    177            1   
mid                          64   10205                    170            0   
premium                      44     775                     93            0   

genre          Early Access  ...  Short  Simulation  Software Training  \
price_segment                ...                                         
budget                 4284  ...      1       1322